# Phase 4: Ablations & Tuning

1. **Speed-only vs text** — how much does event text help?
2. **Fusion mode** — concat vs add
3. **Hyperparameter sweep** — d_model, num_layers, dropout

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import torch
import itertools

from src import (
    load_train_speeds, load_train_texts, load_test_data,
    load_adjacency,
    build_windows, compute_norm_stats, normalize, denormalize,
    train_val_split, compute_mse, write_submission,
)
from src.model import TrafficGNN
from src.train import (
    build_adj_tensor, encode_texts, create_dataloaders, train_model,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
s1, s2 = load_train_speeds()
t1, t2 = load_train_texts()
adj = load_adjacency()

X1, T1, Y1 = build_windows(s1, t1)
X2, T2, Y2 = build_windows(s2, t2)

X_all = np.concatenate([X1, X2], axis=0).astype(np.float32)
Y_all = np.concatenate([Y1, Y2], axis=0).astype(np.float32)
T_all = np.array(T1 + T2)

mean, std = compute_norm_stats(np.concatenate([s1, s2], axis=0))
X_norm = normalize(X_all, mean, std)
Y_norm = normalize(Y_all, mean, std)

X_tr, _, Y_tr, X_va, _, Y_va = train_val_split(X_norm, None, Y_norm, val_frac=0.2)
T_tr = T_all[:len(X_tr)]
T_va = T_all[len(X_tr):]

print(f"Train: {X_tr.shape}, Val: {X_va.shape}")

T_emb_tr = encode_texts(T_tr.tolist())
T_emb_va = encode_texts(T_va.tolist())
adj_t = build_adj_tensor(adj).to(DEVICE)

In [ ]:
@torch.no_grad()
def predict_and_score(model, X, T_emb, adj_t, Y, device):
    model.eval()
    preds = []
    bs = 64
    for i in range(0, len(X), bs):
        xb = torch.tensor(X[i:i+bs], dtype=torch.float32, device=device)
        tb = torch.tensor(T_emb[i:i+bs], dtype=torch.float32, device=device)
        pb = model(xb, tb, adj_t)
        preds.append(pb.cpu().numpy())
    yp_norm = np.concatenate(preds, axis=0)
    yp = denormalize(yp_norm, mean, std)
    yt = denormalize(Y, mean, std)
    return compute_mse(yp, yt), yp


def run_experiment(use_text, fusion="add", d_model=64, num_layers=4, dropout=0.2,
                   lr=1e-3, epochs=80, patience=12):
    tl, vl = create_dataloaders(X_tr, T_emb_tr, Y_tr, X_va, T_emb_va, Y_va, batch_size=32)
    model = TrafficGNN(
        num_nodes=1260, in_steps=15, out_steps=3,
        text_dim=384, d_model=d_model, num_layers=num_layers,
        dropout=dropout, use_text=use_text, fusion=fusion,
    ).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    train_model(model, tl, vl, adj_t, DEVICE, epochs=epochs, lr=lr,
                patience=patience, verbose=False)
    mse, _ = predict_and_score(model, X_va, T_emb_va, adj_t, Y_va, DEVICE)
    return mse, n_params

## Experiment 1: Speed-only vs Text

In [ ]:
results = {}

mse_no_text, n1 = run_experiment(use_text=False)
results["speed-only"] = mse_no_text
print(f"Speed-only:         MSE={mse_no_text:.4f}  ({n1:,} params)")

mse_text_add, n2 = run_experiment(use_text=True, fusion="add")
results["text (add)"] = mse_text_add
print(f"Text (add fusion):  MSE={mse_text_add:.4f}  ({n2:,} params)")

mse_text_cat, n3 = run_experiment(use_text=True, fusion="concat")
results["text (concat)"] = mse_text_cat
print(f"Text (concat):      MSE={mse_text_cat:.4f}  ({n3:,} params)")

## Experiment 2: Hyperparameter Sweep

In [ ]:
# Use the best fusion mode from experiment 1
best_fusion = min(results, key=results.get).split("(")[1].rstrip(")") if "(" in min(results, key=results.get) else "add"
print(f"Best fusion: {best_fusion} ({min(results.values()):.4f})")

sweep_results = {}
for d_model in [32, 64, 128]:
    for num_layers in [2, 3, 4, 6]:
        mse, n = run_experiment(
            use_text=True, fusion=best_fusion,
            d_model=d_model, num_layers=num_layers, dropout=0.2,
            epochs=50, patience=10,
        )
        key = f"d={d_model} L={num_layers}"
        sweep_results[key] = (mse, n)
        print(f"{key:<20} MSE={mse:.4f}  ({n:,} params)")

## Summary

In [ ]:
print("=" * 50)
print(f"{'Experiment':<30} {'Val MSE':>10}")
print("-" * 42)
for name, mse in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name:<30} {mse:>10.4f}")

best_sweep = min(sweep_results, key=lambda k: sweep_results[k][0])
print(f"\nBest sweep config: {best_sweep} -> MSE={sweep_results[best_sweep][0]:.4f}")
print(f"Best overall MSE: {min(min(results.values()), sweep_results[best_sweep][0]):.4f}")

## Best Config — Full Retrain & Submit

In [ ]:
# Parse best config
parts = best_sweep.split()
best_d = int(parts[0].split("=")[1])
best_l = int(parts[1].split("=")[1])

T_emb_all = encode_texts(T_all.tolist())

full_ds = torch.utils.data.TensorDataset(
    torch.tensor(X_norm), torch.tensor(T_emb_all), torch.tensor(Y_norm),
)
full_loader = torch.utils.data.DataLoader(full_ds, batch_size=32, shuffle=True)

model_best = TrafficGNN(
    num_nodes=1260, in_steps=15, out_steps=3,
    text_dim=384, d_model=best_d, num_layers=best_l,
    dropout=0.2, use_text=True, fusion=best_fusion,
).to(DEVICE)

opt = torch.optim.Adam(model_best.parameters(), lr=1e-3)
crit = torch.nn.MSELoss()
for epoch in range(1, 81):
    model_best.train()
    total = 0
    for Xb, Tb, Yb in full_loader:
        Xb, Tb, Yb = Xb.to(DEVICE), Tb.to(DEVICE), Yb.to(DEVICE)
        opt.zero_grad()
        loss = crit(model_best(Xb, Tb, adj_t), Yb)
        loss.backward()
        opt.step()
        total += loss.item() * Xb.size(0)
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | loss: {total / len(full_ds):.4f}")

In [ ]:
test_hist, test_texts = load_test_data()
test_hist_norm = normalize(test_hist, mean, std)
test_emb = encode_texts(test_texts)

@torch.no_grad()
def predict_test(model, X, T_emb, adj_t, device):
    model.eval()
    preds = []
    bs = 64
    for i in range(0, len(X), bs):
        xb = torch.tensor(X[i:i+bs], dtype=torch.float32, device=device)
        tb = torch.tensor(T_emb[i:i+bs], dtype=torch.float32, device=device)
        preds.append(model(xb, tb, adj_t).cpu().numpy())
    yp_norm = np.concatenate(preds, axis=0)
    return denormalize(yp_norm, mean, std).clip(min=0)

test_preds = predict_test(model_best, test_hist_norm, test_emb, adj_t, DEVICE)
write_submission(
    test_preds,
    label=f"gnn_d{best_d}_l{best_l}_{best_fusion}",
    models={"gnn": model_best.state_dict()},
)